# 
zaczynamy ostatni


## Podsumowanie — Whitespot Analysis

### Metodologia
- Siatka punktów co 500m: 28,077 lokalizacji
- Filtrowanie: min 800m od sklepu klienta, min 150m od konkurencji, min 500 osób, min 50 budynków
- Po filtrowaniu: 4,837 potencjalnych lokalizacji
- Model: Ridge (CV R²=0.393) na 31 cechach

### Top rekomendacje
| Rank | Obszar | Score | Pop w 500m | Uzasadnienie |
|---|---|---|---|---|
| 1-3 | Katowice/Chorzów (lng ~19.0) | 28-32 | 1k-3.2k | Gęsta zabudowa, duża populacja, brak konkurencji |
| 4-7 | Kraków południe (lng ~19.9-20.0) | 28-29 | 3.5k-5.8k | Wysoka populacja, brak sklepów klienta w pobliżu |

### Ograniczenia
- Model trenowany na 50 próbkach — wyniki traktować jako ranking, nie absolutne wartości
- Foot traffic dla whitespotów oparty na medianach — brak rzeczywistych danych GPS
- Rekomendacja: weryfikacja top lokalizacji przez wizję lokalną

In [1]:
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import folium
from folium.plugins import HeatMap
from shapely import wkt
from shapely.geometry import Point
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import os

load_dotenv()

DATA_DIR      = r"C:\Users\slast\OneDrive\Pulpit\geo_zadanie\01_data\raw"
PROCESSED_DIR = r"C:\Users\slast\OneDrive\Pulpit\geo_zadanie\01_data\procced"

print("✅ Biblioteki załadowane")

✅ Biblioteki załadowane


In [2]:
# Wczytanie features
df = pd.read_csv(os.path.join(PROCESSED_DIR, 'features_extended.csv'))

feature_cols = [c for c in df.columns 
                if c not in ['location_id', 'lat', 'lng', 'monthly_revenue']]

X = df[feature_cols]
y = df['monthly_revenue']

# Trening Ridge na pełnym zbiorze
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = Ridge(alpha=10.0)
model.fit(X_scaled, y)

print(f"✅ Model wytrenowany: Ridge (CV R²=0.393)")
print(f"✅ Features: {len(feature_cols)} cech")

✅ Model wytrenowany: Ridge (CV R²=0.393)
✅ Features: 31 cech


In [3]:
# Obszar analizy
df_area = pd.read_csv(os.path.join(DATA_DIR, 'ds_analysis_area.csv'))
area_geom = wkt.loads(df_area['geometry'].iloc[0])
minx, miny, maxx, maxy = area_geom.bounds

BBOX = {
    'lat_min': miny, 'lat_max': maxy,
    'lng_min': minx, 'lng_max': maxx
}

# Generowanie siatki co ~500m (≈ 0.005 stopnia)
GRID_STEP = 0.005

grid_points = []
for lat in np.arange(miny, maxy, GRID_STEP):
    for lng in np.arange(minx, maxx, GRID_STEP):
        p = Point(lng, lat)
        if area_geom.contains(p):
            grid_points.append({'lat': lat, 'lng': lng})

df_grid = pd.DataFrame(grid_points)
df_grid['location_id'] = [f'GRID_{i:05d}' for i in range(len(df_grid))]

print(f"✅ Siatka: {len(df_grid):,} potencjalnych lokalizacji")

✅ Siatka: 28,077 potencjalnych lokalizacji


In [4]:
# Wczytanie danych przestrzennych
df_competitors = pd.read_csv(os.path.join(DATA_DIR, 'ds_competitors.csv'))
df_poi         = pd.read_csv(os.path.join(DATA_DIR, 'ds_poi.csv'))
df_population  = pd.read_csv(os.path.join(DATA_DIR, 'recruitment_population.csv.gz'), compression='gzip')
df_buildings   = pd.read_csv(os.path.join(DATA_DIR, 'recruitment_buildings.csv.gz'), compression='gzip')
df_locations   = pd.read_csv(os.path.join(DATA_DIR, 'ds_locations.csv'))

# GeoDataFramy w EPSG:2180
gdf_grid = gpd.GeoDataFrame(
    df_grid,
    geometry=gpd.points_from_xy(df_grid['lng'], df_grid['lat']),
    crs='EPSG:4326'
).to_crs('EPSG:2180')

gdf_competitors = gpd.GeoDataFrame(
    df_competitors,
    geometry=gpd.points_from_xy(df_competitors['lng'], df_competitors['lat']),
    crs='EPSG:4326'
).to_crs('EPSG:2180')

gdf_poi = gpd.GeoDataFrame(
    df_poi,
    geometry=gpd.points_from_xy(df_poi['lng'], df_poi['lat']),
    crs='EPSG:4326'
).to_crs('EPSG:2180')

mask = (
    df_population['lat'].between(BBOX['lat_min'], BBOX['lat_max']) &
    df_population['lng'].between(BBOX['lng_min'], BBOX['lng_max'])
)
gdf_population = gpd.GeoDataFrame(
    df_population[mask].copy(),
    geometry=gpd.points_from_xy(df_population[mask]['lng'], df_population[mask]['lat']),
    crs='EPSG:4326'
).to_crs('EPSG:2180')

gdf_buildings = gpd.GeoDataFrame(
    df_buildings,
    geometry=df_buildings['geometry'].apply(wkt.loads),
    crs='EPSG:4326'
).to_crs('EPSG:2180')

gdf_locations = gpd.GeoDataFrame(
    df_locations,
    geometry=gpd.points_from_xy(df_locations['lng'], df_locations['lat']),
    crs='EPSG:4326'
).to_crs('EPSG:2180')

print("✅ Wszystkie GeoDataFramy gotowe")

✅ Wszystkie GeoDataFramy gotowe


In [5]:
BUFFER_RADIUS_M = 500

gdf_grid_buffers = gdf_grid[['location_id', 'geometry']].copy()
gdf_grid_buffers['geometry'] = gdf_grid.geometry.buffer(BUFFER_RADIUS_M)

print("Obliczanie cech dla siatki...")

# 1. Populacja
print("  + demografia...")
joined_pop = gpd.sjoin(gdf_population, gdf_grid_buffers, how='inner', predicate='within')
demo = joined_pop.groupby('location_id').agg(
    pop_total      = ('total', 'sum'),
    pop_households = ('households', 'sum'),
    avg_hh_size    = ('household_size', 'mean')
).reset_index()

# 2. Budynki
print("  + zabudowa...")
joined_bld = gpd.sjoin(gdf_buildings, gdf_grid_buffers, how='inner', predicate='intersects')
residential = ['budynkiMieszkalneJednorodzinne', 'budynkiODwochMieszkaniach',
               'budynkiOTrzechIWiecejMieszkaniach', 'budynkiZbiorowegoZamieszkania']
commercial  = ['budynkiHandlowoUslugowe', 'budynkiBiurowe', 'budynkiPrzemyslowe']
joined_bld['is_residential'] = joined_bld['funogolnabudynku_desc'].isin(residential)
joined_bld['is_commercial']  = joined_bld['funogolnabudynku_desc'].isin(commercial)

bld = joined_bld.groupby('location_id').agg(
    building_count      = ('area', 'count'),
    total_building_area = ('area', 'sum'),
    avg_building_area   = ('area', 'mean'),
    residential_count   = ('is_residential', 'sum'),
    commercial_count    = ('is_commercial', 'sum'),
).reset_index()
bld['building_density'] = (bld['building_count'] / (3.14159 * (BUFFER_RADIUS_M/1000)**2)).round(1)

# 3. POI (bufor 1km)
print("  + POI...")
gdf_grid_buffers_1km = gdf_grid[['location_id', 'geometry']].copy()
gdf_grid_buffers_1km['geometry'] = gdf_grid.geometry.buffer(1000)
joined_poi = gpd.sjoin(gdf_poi, gdf_grid_buffers_1km, how='inner', predicate='within')
poi_pivot = joined_poi.groupby(['location_id', 'category']).size().unstack(fill_value=0)
poi_pivot.columns = [f'poi_{c}' for c in poi_pivot.columns]
poi_pivot['poi_total'] = poi_pivot.sum(axis=1)
poi_pivot = poi_pivot.reset_index()

# 4. Konkurencja
print("  + konkurencja...")
joined_comp = gpd.sjoin(gdf_competitors, gdf_grid_buffers, how='inner', predicate='within')
comp_count = joined_comp.groupby('location_id').size().rename('competitor_count').reset_index()

dist_comp = []
for _, loc in gdf_grid.iterrows():
    d = gdf_competitors.geometry.distance(loc.geometry).min()
    dist_comp.append({'location_id': loc['location_id'], 'dist_nearest_competitor_m': round(d, 1)})
df_dist_comp = pd.DataFrame(dist_comp)

print("✅ Cechy obliczone")

Obliczanie cech dla siatki...
  + demografia...
  + zabudowa...
  + POI...
  + konkurencja...
✅ Cechy obliczone


In [6]:
# Złączenie wszystkich cech
df_grid_features = df_grid[['location_id', 'lat', 'lng']].copy()

for df_part in [demo, bld, poi_pivot, comp_count, df_dist_comp]:
    df_grid_features = df_grid_features.merge(df_part, on='location_id', how='left')

df_grid_features = df_grid_features.fillna(0)

# Cechy pochodne
df_grid_features['residential_ratio'] = (df_grid_features['residential_count'] / df_grid_features['building_count'].replace(0, 1)).round(3)
df_grid_features['commercial_ratio']  = (df_grid_features['commercial_count']  / df_grid_features['building_count'].replace(0, 1)).round(3)
df_grid_features['pop_density']       = (df_grid_features['pop_total'] / (3.14159 * (BUFFER_RADIUS_M/1000)**2)).round(1)

# Foot traffic — brak danych GPS dla nowych lokalizacji — ustawiamy mediany z datasetu treningowego
df_grid_features['signal_count']         = df['signal_count'].median()
df_grid_features['unique_users']         = df['unique_users'].median()
df_grid_features['peak_morning_signals'] = df['peak_morning_signals'].median()
df_grid_features['peak_ratio']           = df['peak_ratio'].median()
df_grid_features['signals_per_user']     = df['signals_per_user'].median()

# Cechy sieci klienta
dist_client = []
for _, loc in gdf_grid.iterrows():
    distances = gdf_locations.geometry.distance(loc.geometry)
    nearest_idx = distances.idxmin()
    dist_client.append({
        'location_id':            loc['location_id'],
        'dist_nearest_client_m':  round(distances.min(), 1),
        'nearest_client_revenue': df_locations.loc[nearest_idx, 'monthly_revenue']
    })
df_dist_client = pd.DataFrame(dist_client)
df_grid_features = df_grid_features.merge(df_dist_client, on='location_id', how='left')

# Odległość do najbliższego POI
dist_poi = []
for _, loc in gdf_grid.iterrows():
    d = gdf_poi.geometry.distance(loc.geometry).min()
    dist_poi.append({'location_id': loc['location_id'], 'dist_nearest_poi_m': round(d, 1)})
df_dist_poi = pd.DataFrame(dist_poi)
df_grid_features = df_grid_features.merge(df_dist_poi, on='location_id', how='left')

# Uzupełnienie brakujących kolumn POI
for cat in ['poi_bank', 'poi_bus_stop', 'poi_hospital', 'poi_mall',
            'poi_park', 'poi_pharmacy', 'poi_restaurant', 'poi_school']:
    if cat not in df_grid_features.columns:
        df_grid_features[cat] = 0

df_grid_features = df_grid_features.fillna(0)

print(f"✅ Finalny grid: {df_grid_features.shape}")

✅ Finalny grid: (28077, 34)


In [7]:
# Upewniamy się że kolumny są w tej samej kolejności co model
X_grid = df_grid_features[feature_cols].fillna(0)
X_grid_scaled = scaler.transform(X_grid)

# Predykcja
df_grid_features['predicted_revenue'] = model.predict(X_grid_scaled)

# Normalizacja do score 0-100
rev_min = df_grid_features['predicted_revenue'].min()
rev_max = df_grid_features['predicted_revenue'].max()
df_grid_features['score'] = (
    (df_grid_features['predicted_revenue'] - rev_min) / (rev_max - rev_min) * 100
).round(1)

print(f"✅ Scoring gotowy")
print(f"Predicted revenue:")
print(f"  min:    {df_grid_features['predicted_revenue'].min():>10,.0f} PLN")
print(f"  mediana:{df_grid_features['predicted_revenue'].median():>10,.0f} PLN")
print(f"  max:    {df_grid_features['predicted_revenue'].max():>10,.0f} PLN")

✅ Scoring gotowy
Predicted revenue:
  min:        14,163 PLN
  mediana:   224,844 PLN
  max:     1,795,370 PLN


In [8]:
# Filtrowanie whitespotów
MIN_DIST_CLIENT_M = 800   # min 800m od istniejącego sklepu klienta
MIN_DIST_COMP_M   = 150   # min 150m od konkurencji

df_whitespots = df_grid_features[
    (df_grid_features['dist_nearest_client_m']     >= MIN_DIST_CLIENT_M) &
    (df_grid_features['dist_nearest_competitor_m'] >= MIN_DIST_COMP_M)
].copy()

print(f"Lokalizacji przed filtrowaniem: {len(df_grid_features):,}")
print(f"Lokalizacji po filtrowaniu:     {len(df_whitespots):,}")
print(f"\nTop 10 whitespotów:")
display(df_whitespots.nlargest(10, 'score')[
    ['location_id', 'lat', 'lng', 'score', 'predicted_revenue',
     'pop_total', 'building_count', 'competitor_count']
].round(1))

Lokalizacji przed filtrowaniem: 28,077
Lokalizacji po filtrowaniu:     27,598

Top 10 whitespotów:


,location_id,lat,lng,score,predicted_revenue,pop_total,building_count,competitor_count
25761,GRID_25761,50.4,18.6,100.0,1795369.8,0.0,1.0,0.0
16848,GRID_16848,50.3,18.7,57.5,1037592.7,0.0,7.0,0.0
17155,GRID_17155,50.3,18.7,50.2,908338.9,0.0,6.0,0.0
16643,GRID_16643,50.3,19.2,49.0,886498.4,0.0,11.0,0.0
16642,GRID_16642,50.3,19.2,48.0,868450.8,0.0,14.0,0.0
16592,GRID_16592,50.3,18.9,47.1,852508.9,0.0,3.0,0.0
23468,GRID_23468,50.4,19.3,47.0,850846.5,0.0,1.0,0.0
17156,GRID_17156,50.3,18.7,46.1,834971.3,0.0,11.0,0.0
16847,GRID_16847,50.3,18.7,44.3,802636.7,0.0,7.0,0.0
16591,GRID_16591,50.3,18.9,43.2,784496.9,0.0,4.0,0.0


In [9]:
MIN_DIST_CLIENT_M = 800
MIN_DIST_COMP_M   = 150
MIN_POP_TOTAL     = 500    # min 500 osób w buforze
MIN_BUILDINGS     = 50     # min 50 budynków

df_whitespots = df_grid_features[
    (df_grid_features['dist_nearest_client_m']     >= MIN_DIST_CLIENT_M) &
    (df_grid_features['dist_nearest_competitor_m'] >= MIN_DIST_COMP_M)   &
    (df_grid_features['pop_total']                 >= MIN_POP_TOTAL)     &
    (df_grid_features['building_count']            >= MIN_BUILDINGS)
].copy()

print(f"Lokalizacji przed filtrowaniem: {len(df_grid_features):,}")
print(f"Lokalizacji po filtrowaniu:     {len(df_whitespots):,}")
print(f"\nTop 10 whitespotów:")
display(df_whitespots.nlargest(10, 'score')[
    ['location_id', 'lat', 'lng', 'score', 'predicted_revenue',
     'pop_total', 'building_count', 'competitor_count']
].round(1))

Lokalizacji przed filtrowaniem: 28,077
Lokalizacji po filtrowaniu:     4,837

Top 10 whitespotów:


,location_id,lat,lng,score,predicted_revenue,pop_total,building_count,competitor_count
16914,GRID_16914,50.3,19.0,32.2,587261.9,3232.0,774.0,0.0
16913,GRID_16913,50.3,19.0,31.4,573330.5,3020.0,595.0,0.0
17221,GRID_17221,50.3,19.0,30.6,558701.0,987.0,317.0,0.0
4166,GRID_04166,50.1,20.0,29.3,536314.7,3561.0,378.0,0.0
17222,GRID_17222,50.3,19.0,29.1,532387.6,3073.0,403.0,0.0
4165,GRID_04165,50.1,19.9,28.7,525478.1,3905.0,610.0,0.0
4164,GRID_04164,50.1,19.9,28.6,524287.1,5851.0,797.0,0.0
17527,GRID_17527,50.3,19.0,28.6,524109.7,1242.0,265.0,0.0
16606,GRID_16606,50.3,19.0,28.3,518264.2,7096.0,1093.0,1.0
17220,GRID_17220,50.3,19.0,28.3,517804.7,1732.0,362.0,0.0


In [10]:
center_lat = (BBOX['lat_min'] + BBOX['lat_max']) / 2
center_lng = (BBOX['lng_min'] + BBOX['lng_max']) / 2

m = folium.Map(location=[center_lat, center_lng], zoom_start=10, tiles='CartoDB positron')

# Heatmapa whitespotów
heat_data = df_whitespots[['lat', 'lng', 'score']].values.tolist()
HeatMap(heat_data, radius=15, blur=10, min_opacity=0.3,
        gradient={0.4: 'blue', 0.65: 'lime', 1: 'red'}).add_to(m)

# Istniejące sklepy klienta
for _, row in df_locations.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lng']],
        radius=8, color='#1a6bc9', fill=True, fill_opacity=0.9,
        tooltip=f"Klient: {row['location_id']}<br>Przychód: {row['monthly_revenue']:,.0f} PLN"
    ).add_to(m)

# Konkurencja
df_competitors = pd.read_csv(os.path.join(DATA_DIR, 'ds_competitors.csv'))
for _, row in df_competitors.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lng']],
        radius=4, color='#e74c3c', fill=True, fill_opacity=0.7,
        tooltip=f"{row['brand']}"
    ).add_to(m)

# TOP 10 whitespotów
top10 = df_whitespots.nlargest(10, 'score')
for rank, (_, row) in enumerate(top10.iterrows(), 1):
    folium.Marker(
        location=[row['lat'], row['lng']],
        icon=folium.DivIcon(
            html=f'<div style="background:#2ecc71;color:white;border-radius:50%;'
                 f'width:26px;height:26px;text-align:center;line-height:26px;'
                 f'font-weight:bold;font-size:12px;border:2px solid white">{rank}</div>',
            icon_size=(26, 26), icon_anchor=(13, 13)
        ),
        tooltip=f"TOP {rank} | Score: {row['score']:.1f} | Pop: {row['pop_total']:.0f} | Pred: {row['predicted_revenue']:,.0f} PLN"
    ).add_to(m)

# Legenda
legend_html = """
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;background:white;
     padding:12px;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,0.3);font-size:13px">
  <b>Legenda</b><br>
  <span style="color:#1a6bc9">●</span> Sklep klienta<br>
  <span style="color:#e74c3c">●</span> Konkurencja<br>
  <span style="color:#2ecc71"><b>①</b></span> TOP 10 Whitespot<br>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

output_path = r"C:\Users\slast\OneDrive\Pulpit\geo_zadanie\whitespot_map.html"
m.save(output_path)
print(f"✅ Mapa zapisana: {output_path}")
m

✅ Mapa zapisana: C:\Users\slast\OneDrive\Pulpit\geo_zadanie\whitespot_map.html


## Podsumowanie — Whitespot Analysis

### Metodologia
- Siatka punktów co 500m: 28,077 lokalizacji
- Filtrowanie: min 800m od sklepu klienta, min 150m od konkurencji, min 500 osób, min 50 budynków
- Po filtrowaniu: 4,837 potencjalnych lokalizacji
- Model: Ridge (CV R²=0.393) na 31 cechach

### Top rekomendacje
| Rank | Obszar | Score | Pop w 500m | Uzasadnienie |
|---|---|---|---|---|
| 1-3 | Katowice/Chorzów (lng ~19.0) | 28-32 | 1k-3.2k | Gęsta zabudowa, duża populacja, brak konkurencji |
| 4-7 | Kraków południe (lng ~19.9-20.0) | 28-29 | 3.5k-5.8k | Wysoka populacja, brak sklepów klienta w pobliżu |

### Ograniczenia
- Model trenowany na 50 próbkach — wyniki traktować jako ranking, nie absolutne wartości
- Foot traffic dla whitespotów oparty na medianach — brak rzeczywistych danych GPS
- Rekomendacja: weryfikacja top lokalizacji przez wizję lokalną

In [12]:
from geopy.geocoders import Nominatim
import time

geolocator = Nominatim(user_agent="dataplace_whitespot")

top10 = df_whitespots.nlargest(10, 'score').reset_index(drop=True)

addresses = []
for i, row in top10.iterrows():
    location = geolocator.reverse(f"{row['lat']}, {row['lng']}", language='pl')
    addresses.append({
        'rank':               i + 1,
        'lat':                row['lat'],
        'lng':                row['lng'],
        'score':              row['score'],
        'predicted_revenue':  row['predicted_revenue'],
        'pop_total':          row['pop_total'],
        'adres':              location.address if location else 'brak'
    })
    time.sleep(1)  # limit zapytań Nominatim

df_addresses = pd.DataFrame(addresses)
display(df_addresses[['rank', 'adres', 'score', 'predicted_revenue', 'pop_total']].round(0))

,rank,adres,score,predicted_revenue,pop_total
0,1,"31, 3 Maja, Załęskie Przedmieście, Śródmieście...",32.0,587262.0,3232.0
1,2,"9, 5, Sądowa, Załęskie Przedmieście, Śródmieśc...",31.0,573331.0,3020.0
2,3,"Friedricha Wilhelma Grundmanna, Załęskie Przed...",31.0,558701.0,987.0
3,4,"Bulwar Kurlandzki, Kazimierz, Stare Miasto, Kr...",29.0,536315.0,3561.0
4,5,"Szkolne Schronisko Młodzieżowe Ślązaczek, 26, ...",29.0,532388.0,3073.0
5,6,"Świętego Wawrzyńca, Kazimierz, Stare Miasto, K...",29.0,525478.0,3905.0
6,7,"Kościół pw. Bożego Ciała, Świętego Wawrzyńca, ...",29.0,524287.0,5851.0
7,8,"eSmoking World, Chorzowska, Osiedle Ducha, Dąb...",29.0,524110.0,1242.0
8,9,"15, Plebiscytowa, Śródmieście, Katowice, Górno...",28.0,518264.0,7096.0
9,10,"Techvitas Polska, 15B, Żelazna, Dąb, Katowice,...",28.0,517805.0,1732.0


In [13]:
df_addresses.to_csv(
    r"C:\Users\slast\OneDrive\Pulpit\geo_zadanie\01_data\procced\top10_whitespots.csv",
    index=False
)
print("✅ Zapisano top10_whitespots.csv")

✅ Zapisano top10_whitespots.csv


In [14]:

df_whitespots.to_csv(
    r"C:\Users\slast\OneDrive\Pulpit\geo_zadanie\01_data\procced\whitespot_scores.csv",
    index=False
)
print(f"✅ whitespot_scores.csv zapisany: {len(df_whitespots):,} lokalizacji")

✅ whitespot_scores.csv zapisany: 4,837 lokalizacji
